# Phase 2: Data Cleaning & Deduplication

Once we have crawled raw articles, the next crucial step is **Deduplication**. News articles in Vietnam are frequently syndicated, cross-posted, or rephrased across different publishers. To avoid leakage and redundancy in downstream tasks, our cleaning pipeline employs a two-tier deduplication workflow:

1. **Exact Deduplication:** Relies on URL matching, canonical URL checking, and normalized title comparison.
2. **Semantic Deduplication:** Uses sentence embeddings to detect paraphrased headlines that have a cosine similarity above a specified threshold.

Let's walk through these stages.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
import yaml

# Ensure project root is in system path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.dedup.exact_dedup import ExactDeduplicator
print("Deduplication modules imported successfully.")

Deduplication modules imported successfully.


## 1. Text Normalization & Exact Match Checking
Exact title deduplication uses standard normalization. We convert text to lowercase, strip punctuation, and collapse multiple whitespaces.

In [2]:
deduper = ExactDeduplicator(config_path="../configs/dedup_config.yaml")

sample_titles = [
    "Giá vàng hôm nay 02/06: Tăng điên cuồng chạm đỉnh mới!",
    "Giá vàng hôm nay 02-06: Tăng điên cuồng, chạm đỉnh mới...",
    "giá vàng hôm nay 02/06  tăng điên cuồng chạm đỉnh mới"
]

print("Normalized outputs:")
for t in sample_titles:
    print(f"Original:  {t}")
    print(f"Normalized: {deduper.normalize_text(t)}")
    print("-" * 40)

Normalized outputs:
Original:  Giá vàng hôm nay 02/06: Tăng điên cuồng chạm đỉnh mới!
Normalized: giá vàng hôm nay 02 06 tăng điên cuồng chạm đỉnh mới
----------------------------------------
Original:  Giá vàng hôm nay 02-06: Tăng điên cuồng, chạm đỉnh mới...
Normalized: giá vàng hôm nay 02 06 tăng điên cuồng chạm đỉnh mới
----------------------------------------
Original:  giá vàng hôm nay 02/06  tăng điên cuồng chạm đỉnh mới
Normalized: giá vàng hôm nay 02 06 tăng điên cuồng chạm đỉnh mới
----------------------------------------


## 2. Semantic Deduplication Using Embeddings
Syndicated content may change the wording slightly (e.g. replacing words with synonyms or swapping phrase order). To resolve this, we map titles into vector space and compute cosine similarity.

We use the multilingual pre-trained model: `paraphrase-multilingual-MiniLM-L12-v2`.

In [3]:
# Load deduplication config
with open("../configs/dedup_config.yaml", "r", encoding="utf-8") as f:
    dedup_config = yaml.safe_load(f)
    
print("Semantic Deduplication Config thresholds:")
print(f"  - Auto-remove threshold: {dedup_config['thresholds']['auto_remove']} (Exact duplicates & rephrases)")
print(f"  - Probable-duplication threshold: {dedup_config['thresholds']['probable_dup']} (High risk of duplication)")
print(f"  - Keep/Review candidate threshold: {dedup_config['thresholds']['keep_threshold']}")

Semantic Deduplication Config thresholds:
  - Auto-remove threshold: 0.97 (Exact duplicates & rephrases)
  - Probable-duplication threshold: 0.92 (High risk of duplication)
  - Keep/Review candidate threshold: 0.8


Let's simulate a semantic deduplication task using local embeddings or a mock cosine similarity function.

In [4]:
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"Loading model: {model_name} on CPU...")
model = SentenceTransformer(model_name, device="cpu")

headlines = [
    "Giá xăng tiếp tục tăng mạnh từ chiều nay",
    "Từ chiều nay, giá xăng dầu tiếp tục tăng cao",
    "Vụ tai nạn giao thông nghiêm trọng trên quốc lộ 1A",
]

embeddings = model.encode(headlines)

# Compute cosine similarities
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_0_1 = cosine_sim(embeddings[0], embeddings[1])
sim_0_2 = cosine_sim(embeddings[0], embeddings[2])

print(f"Similarity between Headline 1 & 2: {sim_0_1:.4f} (Expect high - paraphrased)")
print(f"Similarity between Headline 1 & 3: {sim_0_2:.4f} (Expect low - unrelated)")

Loading model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 on CPU...


2026-06-14 19:17:55,995 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-06-14 19:17:55,997 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-06-14 19:17:56,029 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json "HTTP/1.1 200 OK"


2026-06-14 19:17:56,374 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


...

2026-06-14 19:18:04,810 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"


2026-06-14 19:18:04,841 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


2026-06-14 19:18:05,153 - INFO - HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Similarity between Headline 1 & 2: 0.8920 (Expect high - paraphrased)
Similarity between Headline 1 & 3: 0.0991 (Expect low - unrelated)


## 3. Deduplication Stats Summary
Let's see the deduplication results if logs or dataset files are available.

In [5]:
logs_dir = project_root / "logs"
exact_log = logs_dir / "exact_dedup.log"
if exact_log.exists():
    print("Recent Exact Deduplication Log Lines:")
    with open(exact_log, "r", encoding="utf-8") as f:
        lines = f.readlines()
    for line in lines[-10:]:
        print(line.strip())
else:
    print("Exact deduplication log not found. Make sure you have run the deduplication step.")

Recent Exact Deduplication Log Lines:
2026-06-14 19:18:01,147 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-14 19:18:01,178 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-14 19:18:01,469 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-14 19:18:01,501 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-14 19:18:01,794 - INFO - HTTP Request: GET https://huggingface.c